# 01. 문제정의와 설계질문

이번 버전은 사용자 답변을 반영한 설계 정리이다.

목표는 SegFormer scratch 단일 class mask와 640x640 RGB raw 원본 crop을 입력으로 받아, scratch component별 contrast 지표를 계산하고 실제 데이터셋 기반 threshold를 정할 수 있는 후처리 구조를 만드는 것이다.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "scratch_postprocess_utils.py").exists():
    matches = list(Path.cwd().glob("**/Scratch_Postprocess/scratch_postprocess_utils.py"))
    if matches:
        NOTEBOOK_DIR = matches[0].parent
sys.path.insert(0, str(NOTEBOOK_DIR))

from scratch_postprocess_utils import *

ensure_dirs()
print("project:", NOTEBOOK_DIR)

## 확정된 내용

- raw image: RGB
- bit depth: 8-bit
- input size: 640x640 원본 crop
- scale: 1 px = 13 um
- SegFormer mask: scratch 단일 class
- threshold: 아직 미정이며 실제 데이터셋을 근거로 확정
- metric: 하나로 정하지 않고 여러 contrast metric을 비교
- synthetic color pair: 파랑 배경/노랑 scratch, 노랑 배경/빨강 scratch
- scratch geometry: 직선 고정이 아니라 수평/수직/대각선/휘어진 polyline 랜덤 생성

In [ ]:
design = {
    "image": {"size": [640, 640], "channels": "RGB", "bit_depth": 8, "pixel_size_um": PIXEL_SIZE_UM},
    "mask": {"source": "SegFormer", "class_policy": "single scratch binary mask"},
    "threshold_policy": "do not fix threshold from synthetic data; calibrate from real dataset",
    "synthetic_policy": "unlabeled random scratches, not intentionally split into micro/scratch",
    "color_pairs": COLOR_PAIRS,
    "metrics": [
        "luma_contrast_abs",
        "luma_contrast_z",
        "rgb_euclidean_contrast",
        "rgb_contrast_z",
        "max_channel_contrast_abs",
        "mean_channel_contrast_abs",
    ],
}
save_json(RUNS_ROOT / "scratch_postprocess_design_summary.json", design)
design

## 아직 보류하는 내용

- 실제 micro/scratch contrast threshold
- 색상 조합별 threshold를 따로 둘지 여부
- 중간 contrast 영역을 ambiguous로 둘지 여부
- 실제 SegFormer mask가 두께를 과대/과소 추정하는지 여부